In [ ]:
# Cell 1: Hardware / CUDA / BF16 / FP8 Check + Google Drive Persistent Checkpoint Mount
import os
from pathlib import Path

import torch

print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / (1024**3)
    supports_bf16 = props.major >= 8
    supports_fp8 = (props.major == 8 and props.minor == 9) or (props.major >= 9)
    print(f"GPU: {props.name} | VRAM: {vram_gb:.2f} GiB | CC: {props.major}.{props.minor}")
    print(f"BF16 Tensor Cores: {supports_bf16} | Native FP8 (Ada L4 / Hopper): {supports_fp8}")
else:
    print("WARNING: No CUDA GPU detected. Switch Colab Runtime -> Change runtime type -> L4 or A100.")

USE_GOOGLE_DRIVE = True
CKPT_BASE = Path("/content/checkpoints")
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        CKPT_BASE = Path("/content/drive/MyDrive/LucidMachines/checkpoints")
    except Exception as exc:
        print(f"Drive mount skipped ({exc}); using local {CKPT_BASE}")

CKPT_BASE.mkdir(parents=True, exist_ok=True)
os.environ["LUCID_MACHINES_CKPT_DIR"] = str(CKPT_BASE)
print(f"Persistent Checkpoint Root: {CKPT_BASE}")

In [ ]:
# Cell 2: Install Lucid Machines v1.0.0 (Branch: release/lucid-machines-v1.0.0) & Verify Configs
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/TriuneTransformer")
REPO_URL = os.environ.get("LUCID_MACHINES_REPO_URL", "https://github.com/lucid-machines/triune.git")
BRANCH = "release/lucid-machines-v1.0.0"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

%cd /content/TriuneTransformer
!pip install -q --upgrade pip
!pip install -q -e ".[dev]" || pip install -q -e .
!python3 scripts/train.py --model_config junkie-3b --dry-run --batch-size 4 --seq-len 1024 --rank 64 --steer-dirs 4 --use_fp8
!python3 scripts/train.py --model_config junkie-1b --dry-run --batch-size 4 --seq-len 1024 --rank 64 --steer-dirs 4

In [ ]:
# Cell 3: #1 Colab Pro 200-CU Sweet Spot — Train Junkie-3B (2.99B Total / 16 Experts) on L4 24GB FP8
# Why Junkie-3B on Colab L4 (Ada CC 8.9 Native FP8) is the 200-Compute-Unit Sweet Spot:
#   - Total Params: 2,990,355,155 (2.99B) | 16 Routed Experts + 1 Shared Expert (top_k_experts=1)
#   - Active Params: 395.74M MoE-trunk Cortex (469.69M w/ embed) | 206.73M avg dynamic (Reflex -> Limbic -> Cortex)
#   - Resident Footprint: 10.10 GiB static in FP8 (12.89 GiB BF16) -> <20 GiB total resident on L4 24GB!
#     (By contrast, FP32 AdamW requires 33.42 GiB static / 44.56 GiB w/ FP32 master and OOMs on L4 24GB.)
#   - Compute Unit Economics: Colab L4 delivers ~3x more tokens per compute unit than A100, yielding
#     ~21B-23B real tokens (~35B+ AdamW-equivalent tokens via CentroidSteer + Muon & dynamic early exits).
#
# 3-Stage Decontaminated Curriculum:
#   - Stage 1 (80% budget): 50% FineWeb-Edu + 30% Permissive Code + 20% FineMath (foundation pretraining)
#   - Stage 2 (15% budget): High-density textbook/code WSD annealing + joint multi-exit KL calibration
#   - Stage 3 ( 5% budget): Clean decontaminated SFT + Verifiable GRPO RL on executable code/math verifiers
import os
from pathlib import Path

%cd /content/TriuneTransformer
ckpt_3b = Path(os.environ.get("LUCID_MACHINES_CKPT_DIR", "/content/checkpoints")) / "junkie-3b"
ckpt_3b.mkdir(parents=True, exist_ok=True)

CURRICULUM_STAGE = os.environ.get("JUNKIE_CURRICULUM_STAGE", "stage1_foundation")
print(f"Launching Junkie-3B on Colab L4 FP8 | Curriculum Stage: {CURRICULUM_STAGE}")

!python3 scripts/train.py \
    --model_config models/configs/junkie-3b.json \
    --checkpoint_dir "{ckpt_3b}" \
    --auto_resume \
    --use_fp8 \
    --depth_mode dynamic \
    --target_depth_dist 0.50,0.30,0.20 \
    --steps 10000 \
    --batch_size 4 \
    --grad_accum_steps 8 \
    --seq_len 1024 \
    --rank 64 \
    --steer_dirs 4 \
    --steer_mode centroid \
    --save_every 200 \
    --eval_every 200 \
    --log_every 10 \
    --no_wandb \
    --no_hf_login

In [ ]:
# Cell 4: Train Junkie-1B (1.16B Total / 374.99M Expected Active / 317.48M Dynamic Active)
# Exact measured static footprint: 5.87 GiB BF16 (4.79 GiB FP8, 0.1992 B/p on experts) vs 12.99 GiB FP32 AdamW.
import os
from pathlib import Path

%cd /content/TriuneTransformer
ckpt_1b = Path(os.environ.get("LUCID_MACHINES_CKPT_DIR", "/content/checkpoints")) / "junkie-1b"
ckpt_1b.mkdir(parents=True, exist_ok=True)

!python3 scripts/train.py \
    --model_config models/configs/junkie-1b.json \
    --checkpoint_dir "{ckpt_1b}" \
    --auto_resume \
    --use_fp8 \
    --depth_mode dynamic \
    --steps 8000 \
    --batch_size 4 \
    --grad_accum_steps 8 \
    --seq_len 1024 \
    --rank 64 \
    --steer_dirs 4 \
    --steer_mode centroid \
    --save_every 200 \
    --eval_every 200 \
    --log_every 10 \
    --no_wandb \
    --no_hf_login

In [ ]:
# Cell 5: Train Junkie-Mini (535.10M Total / 230.81M Expected Active) for Rapid Single-GPU Iteration
# Fits in ~3.12 GiB static VRAM (CentroidSteer rank=64: 0.1992 B/p on experts) — runs on L4, T4, or A100.
import os
from pathlib import Path

%cd /content/TriuneTransformer
ckpt_mini = Path(os.environ.get("LUCID_MACHINES_CKPT_DIR", "/content/checkpoints")) / "junkie-mini"
ckpt_mini.mkdir(parents=True, exist_ok=True)

!python3 scripts/train.py \
    --model_config models/configs/junkie-mini.json \
    --checkpoint_dir "{ckpt_mini}" \
    --auto_resume \
    --steps 5000 \
    --batch_size 4 \
    --grad_accum_steps 8 \
    --seq_len 1024 \
    --rank 64 \
    --steer_dirs 4 \
    --steer_mode centroid \
    --save_every 200 \
    --eval_every 200 \
    --log_every 10 \
    --no_wandb \
    --no_hf_login

In [ ]:
# Cell 6: Train Flagship Junkie-7B (7.40B Total / 1.95B Expected Active) on Single A100-80GB
# Exact measured static footprint with CentroidSteer (rank=256, 0.3678 B/p on experts):
#   BF16 Weights (13.78 GiB) + BF16 Grads (13.78 GiB) + CentroidSteer+Muon+Base Opt (6.94 GiB) = 34.49 GiB static
#   (Leaves ~45.5 GiB headroom on 80GB A100! By contrast, FP32 AdamW at 8 B/p states needs 82.67 GiB static and OOMs.)
# For Colab A100-40GB or L4 (24GB), automatically enables `--streaming`.
import os
from pathlib import Path

import torch

%cd /content/TriuneTransformer
ckpt_7b = Path(os.environ.get("LUCID_MACHINES_CKPT_DIR", "/content/checkpoints")) / "junkie-7b"
ckpt_7b.mkdir(parents=True, exist_ok=True)

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0.0
extra_flag = "--streaming" if vram_gb < 45.0 else ""
print(f"Detected VRAM: {vram_gb:.1f} GiB | Extra flag: {extra_flag or 'none (full A100-80GB resident)'}")

!python3 scripts/train.py \
    --model_config models/configs/junkie-7b.json \
    --checkpoint_dir "{ckpt_7b}" \
    --auto_resume \
    --steps 2000 \
    --batch_size 1 \
    --grad_accum_steps 16 \
    --seq_len 1024 \
    --rank 256 \
    --steer_dirs 4 \
    --steer_mode centroid \
    --save_every 100 \
    --eval_every 200 \
    --log_every 10 \
    --no_wandb \
    --no_hf_login \
    {extra_flag}

In [ ]:
# Cell 7: CentroidSteer 5-Seed FineWeb-Edu Scaling Validation vs Scientific Baselines
# Compares CentroidSteer against No-Steer Low-Rank, Memory-Matched Random Subspace, and Full-Rank AdamW.
%cd /content/TriuneTransformer
!python3 benchmarks/validate_centroid_steer_upgrade.py \
    --seeds 42 1337 2026 777 1024 \
    --md-steps 120 \
    --fw-steps 250 \
    --output-json benchmarks/colab_centroidsteer_results.json \
    --output-report benchmarks/COLAB_CENTROIDSTEER_REPORT.md

In [ ]:
# Cell 8: Launch Polysom Studio v1.0.0 Live on Cloud GPU (Colab Port Forward / Window / Iframe)
import subprocess
import time
import urllib.request

%cd /content/TriuneTransformer
studio_proc = subprocess.Popen(
    ["kcx", "serve", "--host", "127.0.0.1", "--port", "8000"],
    stdout=open("/tmp/polysom_studio.log", "w"),
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/v1/telemetry/gpu", timeout=1) as r:
            if r.status == 200:
                break
    except Exception:
        time.sleep(1)

try:
    from google.colab import output as colab_output

    proxy_url = colab_output.eval_js("google.colab.kernel.proxyPort(8000)")
    print(f"Polysom Studio v1.0.0 live proxy URL: {proxy_url}")
    colab_output.serve_kernel_port_as_window(8000)
    colab_output.serve_kernel_port_as_iframe(8000, height="760")
except Exception as exc:
    print(f"Polysom Studio v1.0.0 running at http://127.0.0.1:8000 (Colab proxy unavailable: {exc})")